# HoTHP vs RoTHP: β_norm Sweep

**Run on Colab: Runtime → Change runtime type → T4 GPU**

---

## Question

The horizon extrapolation experiment showed that HoTHP's monotonic decay advantage
is **large and significant** for fast-decay processes (β_norm≈0.37) and **null** for
slow-decay processes (β_norm≈0.025). This notebook characterises the **transition**:

> At what β_norm does HoTHP start outperforming RoTHP on OOD temporal lags?

We fix mu and alpha (same Hawkes excitation structure) and sweep **beta** (the decay
rate of the triggering kernel), which directly controls β_norm = beta × mean_gap.

## Expected outcome

- **Low β_norm** (slow decay): distant events still matter → RoTHP can approximate
  the right weights; both models degrade similarly. HoTHP advantage ≈ 0.
- **High β_norm** (fast decay): distant events should be ignored → HoTHP's monotonic
  kernel enforces this by construction; RoTHP may assign high attention to OOD lags
  due to sinusoidal oscillation. HoTHP advantage > 0, growing with factor.

The transition point and its sharpness are the empirical findings.

---

**Expected runtime**: ~60–80 min on T4 GPU  
(7 β values × 5 seeds × 2 models × 1 restart)

In [ ]:
import os
if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
!pip install omegaconf -q

In [ ]:
import os, sys, math, random, hashlib, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ── Sweep parameters ──────────────────────────────────────────────────────
TRAIN_LEN      = 50
EXTRAP_FACTORS = [2, 5, 10]
N_SEEDS        = 5     # 5 per β value — sweep is about the curve, not significance at each point
N_RESTARTS     = 1     # 1 restart (vs 2 in horizon experiment) — halves compute for sweep
EPOCHS         = 500
PATIENCE       = 30
BASE_SEED      = 42

# Base process — keep mu/alpha fixed, sweep only beta
# Uses PROC_FAST's parameters so the fast end matches the confirmed result
NUM_TYPES  = 2
PAD_ID     = NUM_TYPES
BASE_MU    = np.array([0.4, 0.4])
BASE_ALPHA = np.array([[0.12, 0.08], [0.08, 0.12]])

# Beta sweep — log-spaced from very slow to very fast decay
# Expected β_norm (beta × mean_gap) roughly: 0.01, 0.03, 0.07, 0.14, 0.25, 0.37, 0.55
BETA_SWEEP = [0.01, 0.05, 0.1, 0.2, 0.35, 0.5, 0.8]
# ─────────────────────────────────────────────────────────────────────────

def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def run_seed(*parts):
    key = '::'.join(map(str, parts))
    return (BASE_SEED + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None: p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

USE_AMP = device.type == 'cuda'
if USE_AMP:
    try:
        _autocast = lambda: torch.amp.autocast(device_type='cuda')
        _Scaler   = torch.amp.GradScaler
    except AttributeError:
        _autocast = torch.cuda.amp.autocast
        _Scaler   = torch.cuda.amp.GradScaler
else:
    _autocast = contextlib.nullcontext
    _Scaler   = None

print(f'Device: {device}  |  AMP: {USE_AMP}')
print(f'Beta sweep: {BETA_SWEEP}')
print(f'Seeds: {N_SEEDS}  |  Restarts: {N_RESTARTS}')

## Data generation

For each beta value, generate train/val/short-test and 2×/5×/10× extrapolation splits.
After generation, measure the **actual β_norm = beta × mean_gap** from training data.
This is the x-axis for all plots.

In [ ]:
def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def make_split(rng, beta, n, horizon, max_ev, min_ev=10):
    return [simulate_hawkes(rng, BASE_MU, BASE_ALPHA, beta,
                            horizon, min_ev, max_ev) for _ in range(n)]


# Generate all data, measure actual β_norm per beta value
rng = np.random.default_rng(run_seed('sweep', 'data'))
sweep_data   = {}   # beta → split dict
actual_bnorm = {}   # beta → measured β_norm

print('Generating data for each beta value...')
print(f'{"beta":>8}  {"mean_gap":>10}  {"β_norm":>8}  {"influence@10x":>14}')
for beta in BETA_SWEEP:
    splits = {
        'train': make_split(rng, beta, 500, horizon=50.0,       max_ev=TRAIN_LEN),
        'val':   make_split(rng, beta, 150, horizon=50.0,       max_ev=TRAIN_LEN),
        'short': make_split(rng, beta, 200, horizon=50.0,       max_ev=TRAIN_LEN),
    }
    for f in EXTRAP_FACTORS:
        splits[f'extrap_{f}x'] = make_split(
            rng, beta, 200,
            horizon=50.0 * f, max_ev=TRAIN_LEN * f, min_ev=TRAIN_LEN + 5)
    sweep_data[beta] = splits

    # Measure actual β_norm from training sequences
    gaps = []
    for seq in splits['train']:
        ts = sorted([t for t, _ in seq])
        gaps.extend([ts[i] - ts[i-1] for i in range(1, len(ts))])
    mg = float(np.mean(gaps))
    bn = beta * mg
    actual_bnorm[beta] = bn
    infl = math.exp(-bn * (TRAIN_LEN * max(EXTRAP_FACTORS) - 1))
    print(f'{beta:>8.3f}  {mg:>10.4f}  {bn:>8.4f}  {infl:>14.6f}')

print('\nβ_norm values (x-axis for plots):')
print([f'{actual_bnorm[b]:.3f}' for b in BETA_SWEEP])

In [ ]:
def to_tensors(seqs):
    """Per-sequence prefix normalization — same as HoTHP's internal _normalize_timestamps."""
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad  = torch.zeros(B, L)
    d_pad  = torch.zeros(B, L)
    k_pad  = torch.full((B, L), PAD_ID, dtype=torch.long)
    npm    = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl]   = 1.0
        m = causal.clone(); m[:, sl:] = True; m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(data, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, generator=g)


# Pre-convert all splits to tensors once
print('Converting sequences to tensors...')
sweep_tensors = {}
for beta in BETA_SWEEP:
    sweep_tensors[beta] = {k: to_tensors(v) for k, v in sweep_data[beta].items()}
print('Done.')

In [ ]:
config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'BetaSweep',
    'thinning': {'num_sample':1,'num_exp':500,'over_sample_rate':5.0,
                 'patience_counter':5,'num_samples_boundary':5,'dtime_max':5.0,'num_step_gen':1},
    'loss_integral_num_sample_per_step': 20, 'use_mc_samples': False,
})


def eval_nll(model, dl):
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            with _autocast():
                l, n = model.loglike_loss(batch)
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def eval_ood_nll(model, dl, cutoff=TRAIN_LEN):
    model.eval(); total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            t, d, k, npm, attn = [x.to(device) for x in batch]
            ood_mask = npm.clone()
            ood_mask[:, :cutoff] = 0.0
            if ood_mask.sum() == 0: continue
            with _autocast():
                l, n = model.loglike_loss([t, d, k, ood_mask, attn])
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def train_once(model, train_dl, val_dl, lr):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
    scaler = _Scaler(enabled=True) if USE_AMP else None
    best_val, best_state, no_imp = float('inf'), None, 0
    for ep in range(EPOCHS):
        model.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast():
                l, n = model.loglike_loss(batch)
                nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                if scaler:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
        val = eval_nll(model, val_dl)
        sched.step(val)
        if val < best_val - 1e-4:
            best_val = val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    if best_state: model.load_state_dict(best_state)
    return best_val


def train_best_of(cls, train_dl, val_dl, lr, base_seed):
    best_val, best_state = float('inf'), None
    for r in range(N_RESTARTS):
        set_seed(base_seed + r * 7919)
        m = cls(config).to(device)
        v = train_once(m, train_dl, val_dl, lr)
        if v < best_val:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
    m = cls(config).to(device)
    m.load_state_dict(best_state)
    return m, best_val


print('Model config and training utilities ready.')

## Main sweep

For each β value and each seed:
1. Train RoTHP and HoTHP on short sequences (max lag ≈ 49)
2. Evaluate OOD NLL on 2×/5×/10× extrapolation sets

Outer loop = beta values; inner loop = seeds.

In [ ]:
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
results = []  # one dict per (beta, seed, factor)

for beta in BETA_SWEEP:
    bn = actual_bnorm[beta]
    tensors = sweep_tensors[beta]

    val_dl   = make_loader(tensors['val'],   64)
    short_dl = make_loader(tensors['short'], 64)
    extrap_dls = {f: make_loader(tensors[f'extrap_{f}x'], 16) for f in EXTRAP_FACTORS}

    print(f'\n{"="*60}')
    print(f'beta={beta:.3f}  β_norm={bn:.4f}')
    print('='*60)

    for seed_idx, seed in enumerate(seeds):
        print(f'  Seed {seed_idx+1}/{N_SEEDS} (seed={seed})', end='  ')

        train_dl = make_loader(tensors['train'], 64, shuffle=True,
                               seed=run_seed(beta, seed))

        rothp, rv = train_best_of(RoTHP, train_dl, val_dl, lr=1e-3,
                                   base_seed=run_seed(beta, 'rothp', seed))
        hothp, hv = train_best_of(HoTHP, train_dl, val_dl, lr=5e-4,
                                   base_seed=run_seed(beta, 'hothp', seed))

        r_short = eval_nll(rothp, short_dl)
        h_short = eval_nll(hothp, short_dl)
        print(f'val RoTHP={rv:.4f} HoTHP={hv:.4f}')

        for f in EXTRAP_FACTORS:
            edl   = extrap_dls[f]
            r_ood = eval_ood_nll(rothp, edl)
            h_ood = eval_ood_nll(hothp, edl)
            results.append({
                'beta': beta, 'bnorm': bn,
                'seed': seed, 'factor': f,
                'r_short': r_short, 'h_short': h_short,
                'r_ood': r_ood,     'h_ood': h_ood,
                'adv_ood': r_ood - h_ood,          # positive = HoTHP wins
                'r_ood_deg': r_ood - r_short,
                'h_ood_deg': h_ood - h_short,
            })

print('\nSweep complete.')

## Results and plots

In [ ]:
from scipy import stats

df = pd.DataFrame(results)
bnorm_vals = sorted(df['bnorm'].unique())

print('=' * 75)
print('RESULTS: β_norm sweep')
print('=' * 75)
print(f'OOD zone = events at positions >= {TRAIN_LEN}  |  Seeds: {N_SEEDS}')
print()
print(f'{"β_norm":>8}  {"factor":>7}  {"adv_ood (mean)": >16}  {"±std":>8}  {"p_ood":>7}  verdict')

for bn in bnorm_vals:
    sub = df[df['bnorm'].round(4) == round(bn, 4)]
    beta_val = sub['beta'].iloc[0]
    print(f'  beta={beta_val:.3f}  β_norm={bn:.4f}')
    for f in EXTRAP_FACTORS:
        row = sub[sub['factor'] == f]
        adv = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2/2 if t_stat > 0 else 1 - p2/2
        sig = '✓' if p1 < 0.05 else ('~' if p1 < 0.10 else '✗')
        winner = 'HoTHP' if adv.mean() > 0 else 'RoTHP '
        print(f'    {f}x: {adv.mean():>+.4f} ± {adv.std():.4f}   p={p1:.3f}  {sig}  {winner}')

In [ ]:
# ── Plot 1: OOD NLL curves — the clearest picture of the divergence ───────
#
# Each panel = one extrapolation factor.
# x-axis = β_norm (actual, measured from data)
# y-axis = OOD NLL
# Blue = RoTHP (expected to rise with β_norm)
# Red  = HoTHP (expected to stay flat)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
colors = {'RoTHP': '#4C72B0', 'HoTHP': '#C44E52'}

for ax, f in zip(axes, EXTRAP_FACTORS):
    sub = df[df['factor'] == f]
    grp = sub.groupby('bnorm')

    for model_col, label in [('r_ood', 'RoTHP'), ('h_ood', 'HoTHP')]:
        m = grp[model_col].mean()
        s = grp[model_col].std()
        ax.plot(m.index, m.values, 'o-', color=colors[label],
                lw=2.2, ms=7, label=label)
        ax.fill_between(m.index, m - s, m + s,
                         color=colors[label], alpha=0.15)

    # Mark significance threshold (p < 0.05 for adv_ood)
    for bn_val in bnorm_vals:
        row = sub[sub['bnorm'].round(4) == round(bn_val, 4)]
        adv = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2/2 if t_stat > 0 else 1 - p2/2
        if p1 < 0.05:
            ax.axvline(bn_val, color='green', ls=':', alpha=0.4, lw=1)

    ax.set_xlabel('β_norm (beta × mean_gap)', fontsize=11)
    ax.set_ylabel('OOD NLL', fontsize=11)
    ax.set_title(f'OOD NLL vs β_norm  —  {f}× extrapolation\n'
                 f'(vertical dotted = first p<0.05)', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f'Sweep: how does OOD NLL change with process decay rate?\n'
    f'Train max {TRAIN_LEN} events → test at 2×/5×/10×  |  Seeds: {N_SEEDS}',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('BetaNorm_NLL_Curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: HoTHP advantage curve — the contribution claim ───────────────
#
# x-axis = β_norm
# y-axis = adv_ood = RoTHP_ood_nll − HoTHP_ood_nll  (positive = HoTHP wins)
# Three lines: 2x, 5x, 10x
# Significance markers: * p<0.05, ~ p<0.10

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

factor_colors = {2: '#1f77b4', 5: '#ff7f0e', 10: '#2ca02c'}

# Panel 1: advantage lines
ax = axes[0]
for f in EXTRAP_FACTORS:
    sub = df[df['factor'] == f]
    grp = sub.groupby('bnorm')['adv_ood']
    m, s = grp.mean(), grp.std()
    ax.plot(m.index, m.values, 'o-', color=factor_colors[f],
            lw=2, ms=7, label=f'{f}× extrapolation')
    ax.fill_between(m.index, m - s, m + s,
                     color=factor_colors[f], alpha=0.12)

    # Add significance markers above each point
    for bn_val in bnorm_vals:
        row = sub[sub['bnorm'].round(4) == round(bn_val, 4)]
        adv = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2/2 if t_stat > 0 else 1 - p2/2
        mean_adv = grp.mean()[bn_val]
        std_adv  = grp.std()[bn_val]
        marker = '★' if p1 < 0.05 else ('○' if p1 < 0.10 else '')
        if marker:
            ax.text(bn_val, mean_adv + std_adv + 0.02,
                    marker, ha='center', va='bottom',
                    color=factor_colors[f], fontsize=10)

ax.axhline(0, color='gray', ls='--', alpha=0.6, lw=1.5)
ax.set_xlabel('β_norm (beta × mean_gap)', fontsize=11)
ax.set_ylabel('HoTHP advantage (RoTHP − HoTHP OOD NLL)', fontsize=11)
ax.set_title('HoTHP advantage vs β_norm\n(★ = p<0.05, ○ = p<0.10)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 2: per-seed scatter at 10x (shows variance story)
ax = axes[1]
sub10 = df[df['factor'] == 10]
rng_jit = np.random.RandomState(0)
for x_i, bn_val in enumerate(bnorm_vals):
    adv = sub10[sub10['bnorm'].round(4) == round(bn_val, 4)]['adv_ood'].values
    jitter = rng_jit.uniform(-0.008, 0.008, len(adv))
    ax.scatter([bn_val] * len(adv) + jitter, adv,
               color='#4C72B0', alpha=0.7, s=55, zorder=3)
    ax.plot([bn_val - 0.012, bn_val + 0.012],
            [adv.mean(), adv.mean()],
            color='black', lw=2.5, zorder=4)

ax.axhline(0, color='gray', ls='--', alpha=0.6, lw=1.5)
ax.set_xlabel('β_norm (beta × mean_gap)', fontsize=11)
ax.set_ylabel('HoTHP advantage (10× OOD)', fontsize=11)
ax.set_title('Per-seed scatter at 10× extrapolation\n(bar = mean)', fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'β_norm sweep: when does monotonic decay pay off?  |  Seeds: {N_SEEDS}',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('BetaNorm_Advantage_Curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: OOD degradation (ΔNLL = OOD NLL − short in-dist NLL) ─────────
#
# Shows how much each model degrades as sequences get longer.
# HoTHP should stay near 0 for high β_norm; RoTHP should grow.

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, f in zip(axes, EXTRAP_FACTORS):
    sub = df[df['factor'] == f]
    grp = sub.groupby('bnorm')

    for col, label in [('r_ood_deg', 'RoTHP'), ('h_ood_deg', 'HoTHP')]:
        m = grp[col].mean()
        s = grp[col].std()
        ax.plot(m.index, m.values, 'o-', color=colors[label],
                lw=2.2, ms=7, label=label)
        ax.fill_between(m.index, m - s, m + s,
                         color=colors[label], alpha=0.13)

    ax.axhline(0, color='gray', ls='--', alpha=0.5, lw=1.2,
               label='in-dist baseline')
    ax.set_xlabel('β_norm', fontsize=11)
    ax.set_ylabel('ΔOOD NLL (OOD − short in-dist)', fontsize=11)
    ax.set_title(f'OOD degradation vs β_norm  —  {f}×', fontsize=10)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f'How much does OOD performance degrade with horizon?\n'
    f'ΔNLL = 0 means no degradation from training-dist performance',
    fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('BetaNorm_Degradation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 70)
print('SUMMARY: β_norm transition')
print('=' * 70)
print(f'Training max lag: {TRAIN_LEN - 1}  |  Seeds: {N_SEEDS}')
print()
print('Threshold = lowest β_norm where HoTHP advantage is significant (p<0.05)')
print()

for f in EXTRAP_FACTORS:
    sub = df[df['factor'] == f]
    threshold = None
    print(f'{f}× extrapolation:')
    for bn_val in bnorm_vals:
        row = sub[sub['bnorm'].round(4) == round(bn_val, 4)]
        adv = row['adv_ood'].values
        t_stat, p2 = stats.ttest_1samp(adv, 0)
        p1 = p2/2 if t_stat > 0 else 1 - p2/2
        sig = '✓ p<0.05' if p1 < 0.05 else ('~ p<0.10' if p1 < 0.10 else '✗ ns   ')
        print(f'  β_norm={bn_val:.4f}: adv={adv.mean():+.4f} ± {adv.std():.4f}  {sig}')
        if threshold is None and p1 < 0.05 and adv.mean() > 0:
            threshold = bn_val
    if threshold:
        print(f'  → Threshold: β_norm ≈ {threshold:.4f}')
    else:
        print(f'  → No significant threshold found in this range')
    print()